#**Model - Modus Chora Studio Classification System**

This notebook documents the data cleaning and preprocessing phase of the Sieve Model, a machine learning-driven classification system designed to support personalized legal health assessments for startups and SMEs within the Legal Health Check platform.

The primary objective of the Sieve Model is to classify businesses into relevant sectors and organizational tiers to enable tailored legal guidance and compliance recommendations. In Phase 1, the model focuses on:

1. **Sector Classification (Machine Learning)**: Identifying a company's sector based on its business description, industry classification, and products or services.
2. **Tier Classification (Business Rules)**: Assigning companies to AIKYA HUSTLE, AIKYA GROW, or AIKYA LEAD tiers using predefined employee-count thresholds.
3. ***Cluster Assignment:*** Generating a unique cluster identifier that combines sector and tier classifications.

#**Dataset Description**

The dataset contains startup and SME records collected from publicly available sources. Key attributes include:

1. Company information
2. Business descriptions
3. Industry classifications
4. Employee counts
5. Funding information
6. Products and services
7. Geographic presence
8. Sector labels

***Step 1: Import Needed Python Libraries***

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from google.colab import files
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)
import warnings
warnings.filterwarnings("ignore")

# **File Upload and Dataframe Creation**

In [ ]:
# Upload Excel workbook
uploaded = files.upload()




Saving Raw Data.xlsx to Raw Data.xlsx


In [ ]:
# Load each sheet into a separate DataFrame
company_df = pd.read_excel("Raw Data.xlsx", sheet_name="Dataset")
trigger_words = pd.read_excel("Raw Data.xlsx", sheet_name="Trigger Words")

print("Data loaded successfully.")

Data loaded successfully.


### **Check Dataset Dimensions and info and Previewing Nulls**


In [ ]:

# Preview company dataset
display(company_df.head())
print("Dataset Shape:", company_df.shape)

# Preview trigger word taxonomy
display(trigger_words.head(1))
print("Dataset Shape:", trigger_words.shape)

,COMPANY_ID,COMPANY_NAME,COMPANY_DESCRIPTION,HQ_COUNTRY,HQ_CITY,INDUSTRY,EMPLOYEE_COUNT,YEAR_FOUNDED,PRODUCTS _AND_SERVICES,OPERATING_COUNTRIES,SECTOR(S)
0,COMP-000001,11 Plc,"Downstream petroleum marketing company engaged in fuel retail, lubricants, aviation fuel, and commercial fuel supply.",Nigeria,Lagos,Oil & Gas,"501–1,000",1979,Petrol stations; diesel; gasoline; aviation fuel; lubricants,Nigeria,Oil & Gas; Fuel Retail
1,COMP-000002,3elagi,"Digital pharmacy platform providing online access to medicines, healthcare products, and beauty items with home delivery services.",Egypt,Cairo,E-Pharmacy; Digital Pharmacy,11–50,2017,Online pharmacy; medicine ordering; prescription fulfilment; healthcare products,Egypt,HealthTech; Digital Pharmacy
2,COMP-000003,3Farmate Robotics,Agricultural robotics company designing AI-powered autonomous farming equipment for precision agriculture and crop monitoring.,Ghana,Accra,Agricultural Robotics,2–10,2022,Autonomous farming robots; crop monitoring systems; AI agricultural platforms,Ghana,Manufacturing; Agri-Tech
3,COMP-000004,3lyna,On-demand grocery delivery platform connecting consumers with grocery retailers for home delivery.,Sudan,Khartoum,E-commerce; Grocery Delivery; FoodTech,2–10,2018,Grocery ordering; last-mile delivery; mobile commerce,Sudan,Retail & E-commerce
4,COMP-000005,4-DNA,"Health technology company using genetic intelligence to provide personalized health, fitness, nutrition, and risk insights.",Rwanda,Kigali,Genetics; Preventive Health,1–10,2020,DNA analysis; personalized health insights; genetic risk assessment tools.,Rwanda,HealthTech; Genomics


Dataset Shape: (1475, 11)


,Sieve Sector,Trigger Words/Phrases
0,ATX,"ag-inputs, agri-business, agri-cooperative, agri-food, agri-inputs, agri-processing, agri-tech, agri-tech hardware, agribiotech, agribusiness, agribusiness infrastructure, agribusiness network, agricultural, agricultural biotechnology, agricultural consulting, agricultural data, agricultural drone, agricultural equipment, agricultural equipment technology, agricultural extension, agricultural innovation, agricultural inputs, agricultural logistics, agricultural marketplace, agricultural products, agricultural robotics, agricultural services, agricultural software, agriculture, agriculture & agribusiness, agriculture & agtech, agriculture & biotechnology, agriculture & digital development, agriculture & digital marketplace, agriculture & inputs, agriculture & inputs technology, agriculture & mechanization services, agriculture ai, agriculture ai & software, agriculture iot, agriculture iot & agtech, agriculture saas, agriculture services & technology, agriculture technology, agrifood, agritech, agro, agro-chemicals, agro-industrial energy, agro-industrial processing, agro-nutrition, agro-processing, agrobotics, agrochemical, agroforestry, agronomic insights, agronomist, agronomy, agtech, ai agriculture, alternative protein, animal health, animal husbandry, apiary, apiculture (beekeeping), aquaculture, aquaculture & agtech, aquaponics, beans, beef, beekeeping, biofertilizer, carbon farming, cassava, cattle, chicken, climate-smart agriculture, cocoa, coffee, commercial farming, compost, conservation agriculture, contract farming, controlled environment agriculture, cooperative agtech, cotton, crop, crop advisory, crop intelligence, crop management, crop monitoring, crop production, crop protection, crop yield, crops, cultivation, dairy, digital ag, digital agriculture, digital farming, drip irrigation, dronetech, farm, farm advisory, farm analytics, farm automation, farm cooperative, farm machinery, farm management, farm management system, farm monitoring, farm platform, farm productivity, farm software, farm supply, farm-to-market, farmer, farmer network, farmer training, farmers, farming, farmtech, farmworker, feed, fertiliser, fertilizer, fish farming, fishery, floriculture, fodder, food & beverage, food processing, food production, food security, food supply chain, food systems, food tech, food technology, food value chain, foodtech, forage, forestry, fruits, fungicide, goat, grain, grain milling, greenhouse, greenhouse technology, harvest planning, harvester, harvesting, herbicide, honey, horticulture, hydroponics, input distribution, inputtech, iot agriculture, irrigation, irrigation pump, irrigation system, irrigation technology, irrigationtech, legumes, livestock, livestock management, livestock technology, livestocktech, maize, market linkage, mechanization, mechanized agri, mechanized agriculture, mechanized farming, mega-scale agribusiness, millet, nursery, nutrient management, nutrition, nutrition technology, orchard, organic farming, pest management, pesticide, pet nutrition, pig, plantation, planter, planting, post-harvest, postharvest, poultry, precision ag, precision agri, precision agriculture, precision farming, precision irrigation, precision livestock, precision seeding, producer cooperative, producer organization, regenerative agriculture, regenerative farming, rice, rural agriculture, seed, seed technology, seedling, seeds, seedtech, sheep, smallholder, smallholder farmers, smart ag, smart agriculture, smart farming, smart greenhouse, smart irrigation, soil, soil analytics, soil health, soil testing, solar ag, solar irrigation, solar irrigation equipment, sorghum, specialized nutrition & plant-based food, sprinkler, sugarcane, sustainable agriculture, sustainable food systems, swine, tea, threshing, tillage, tractor, vegetables, vertical farming, weather monitoring, wheat, yield optimization, yield prediction"


Dataset Shape: (8, 2)


In [ ]:
# Display dataset information

company_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1475 entries, 0 to 1474
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0    COMPANY_ID             1475 non-null   object
 1   COMPANY_NAME            1475 non-null   object
 2   COMPANY_DESCRIPTION     1475 non-null   object
 3   HQ_COUNTRY              1475 non-null   object
 4   HQ_CITY                 1474 non-null   object
 5   INDUSTRY                1475 non-null   object
 6   EMPLOYEE_COUNT          1475 non-null   object
 7   YEAR_FOUNDED            1475 non-null   object
 8   PRODUCTS _AND_SERVICES  1475 non-null   object
 9   OPERATING_COUNTRIES     1468 non-null   object
 10  SECTOR(S)               1475 non-null   object
dtypes: object(11)
memory usage: 126.9+ KB


In [ ]:
#standardizing column names

def standardize_column_names(df):
    df.columns = (
        df.columns
            .str.strip()                           # Remove leading/trailing spaces
            .str.replace(r"\s+", "_", regex=True)  # Replace spaces with underscores
            .str.replace(r"_+", "_", regex=True)   # Remove multiple underscores
            .str.replace(r"[()]", "", regex=True)  # Remove parentheses
            .str.lower()                           # Convert to uppercase
    )
    return df

# Apply to both DataFrames
company_df = standardize_column_names(company_df)
trigger_words = standardize_column_names(trigger_words)

print("Column names standardized successfully.")

# Display current column names
print(company_df.columns.tolist())
print(trigger_words.columns.tolist())

Column names standardized successfully.
['company_id', 'company_name', 'company_description', 'hq_country', 'hq_city', 'industry', 'employee_count', 'year_founded', 'products_and_services', 'operating_countries', 'sectors']
['sieve_sector', 'trigger_words/phrases']


In [ ]:

# Convert Columns to Appropriate Data Types

# Convert text columns to string
text_columns = [
    "company_id",
    "company_name",
    "company_description",
    "hq_country",
    "hq_city",
    "industry",
    "employee_count",
    "products_and_services",
    "operating_countries",
    "sectors"
]

company_df[text_columns] = company_df[text_columns].astype("string")

# Convert year founded to numeric
company_df["year_founded"] = pd.to_numeric(
    company_df["year_founded"],
    errors="coerce"
).astype("Int64")

# Convert trigger word table columns to string
trigger_words = trigger_words.astype({
    "sieve_sector": "string",
    "trigger_words/phrases": "string"
})

print("Data types converted successfully.")

Data types converted successfully.


In [ ]:
# Display dataset information

company_df.info()
trigger_words.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1475 entries, 0 to 1474
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   company_id             1475 non-null   string
 1   company_name           1475 non-null   string
 2   company_description    1475 non-null   string
 3   hq_country             1475 non-null   string
 4   hq_city                1474 non-null   string
 5   industry               1475 non-null   string
 6   employee_count         1475 non-null   string
 7   year_founded           1474 non-null   Int64 
 8   products_and_services  1475 non-null   string
 9   operating_countries    1468 non-null   string
 10  sectors                1475 non-null   string
dtypes: Int64(1), string(10)
memory usage: 128.3 KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                

In [ ]:
#standardizing column names

def standardize_column_names(df):
    df.columns = (
        df.columns
            .str.strip()                           # Remove leading/trailing spaces
            .str.replace(r"\s+", "_", regex=True)  # Replace spaces with underscores
            .str.replace(r"_+", "_", regex=True)   # Remove multiple underscores
            .str.replace(r"[()]", "", regex=True)  # Remove parentheses
            .str.lower()                           # Convert to uppercase
    )
    return df

# Apply to both DataFrames
company_df = standardize_column_names(company_df)
trigger_words = standardize_column_names(trigger_words)

print("Column names standardized successfully.")

Column names standardized successfully.


In [ ]:

# Data Validation

print("=" * 70)
print("COMPANY DATASET VALIDATION")
print("=" * 70)

# Dataset dimensions
print(f"\nDataset Shape: {company_df.shape}")

# Data types
print("\nData Types:")
display(company_df.dtypes)

# Missing values
print("\nMissing Values:")
display(company_df.isnull().sum())

# Duplicate records
print(f"\nDuplicate Rows: {company_df.duplicated().sum()}")

# Duplicate Company IDs
print(f"Duplicate Company IDs: {company_df['company_id'].duplicated().sum()}")

# Duplicate Company Names
print(f"Duplicate Company Names: {company_df['company_name'].duplicated().sum()}")

# Summary statistics
print("\nSummary Statistics:")
display(company_df.describe(include="all"))


print("\n\n" + "=" * 70)
print("TRIGGER WORD TAXONOMY VALIDATION")
print("=" * 70)

# Dataset dimensions
print(f"\nDataset Shape: {trigger_words.shape}")

# Data types
print("\nData Types:")
display(trigger_words.dtypes)

# Missing values
print("\nMissing Values:")
display(trigger_words.isnull().sum())

# Duplicate rows
print(f"\nDuplicate Rows: {trigger_words.duplicated().sum()}")

# Duplicate Sieve Sectors
print(f"Duplicate Sieve Sectors: {trigger_words['sieve_sector'].duplicated().sum()}")

# Display taxonomy
print("\nTrigger Word Taxonomy:")
display(trigger_words)

print("\nData validation completed successfully.")

COMPANY DATASET VALIDATION

Dataset Shape: (1475, 11)

Data Types:


,0
company_id,string[python]
company_name,string[python]
company_description,string[python]
hq_country,string[python]
hq_city,string[python]
industry,string[python]
employee_count,string[python]
year_founded,Int64
products_and_services,string[python]
operating_countries,string[python]



Missing Values:


,0
company_id,0
company_name,0
company_description,0
hq_country,0
hq_city,1
industry,0
employee_count,0
year_founded,1
products_and_services,0
operating_countries,7



Duplicate Rows: 0
Duplicate Company IDs: 0
Duplicate Company Names: 0

Summary Statistics:


...


Missing Values:


,0
sieve_sector,0
trigger_words/phrases,0



Duplicate Rows: 0
Duplicate Sieve Sectors: 0

Trigger Word Taxonomy:


sieve_sector  \
0          ATX   
1          ETX   
2          HTX   
3          ERG   
4          MFG   
5          REC   
6          FTX   
7          XSC   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               


Data validation completed successfully.


## Standardization


In [ ]:
# Create a dictionary of trigger words for primary Sieve sectors only
sector_dictionary = {}

for _, row in trigger_words.iterrows():

    sector = row["sieve_sector"].strip().upper()

    # Skip XSC because it is a derived classification
    if sector == "XSC":
        continue

    keywords = [
        keyword.strip().lower()
        for keyword in row["trigger_words/phrases"].split(",")
        if keyword.strip()
    ]

    sector_dictionary[sector] = keywords

In [ ]:
# Assign Standardized Sieve Sector
def assign_sieve_sector(text):

    # Handle missing values
    if pd.isna(text):
        return "Unknown"

    # Convert text to lowercase
    text = str(text).lower()

    # Store matched sectors
    matched_sectors = set()

    # Search for trigger words/phrases
    for sector, keywords in sector_dictionary.items():

        for keyword in keywords:

            # Match the complete word or phrase only
            pattern = r"(?<!\w)" + re.escape(keyword.lower()) + r"(?!\w)"

            if re.search(pattern, text):

                matched_sectors.add(sector)

                # Stop checking this sector once one trigger is found
                break

    # Apply business rules
    if len(matched_sectors) == 0:
        return "Unknown"

    elif len(matched_sectors) == 1:
        return next(iter(matched_sectors))

    else:
        return "XSC"

In [ ]:
# Combine text fields for sector standardization
# Purpose: Create a single searchable field containing the source sector labels and products/services.


company_df["sector_mapping_text"] = (
    company_df["industry"].fillna("") +
    " ; " +
    company_df["sectors"].fillna("")
)

In [ ]:
company_df["sector_mapping_text"].unique()

<StringArray>
[                                                      'Oil & Gas ; Oil & Gas; Fuel Retail',
                              'E-Pharmacy; Digital Pharmacy ; HealthTech; Digital Pharmacy',
                                         'Agricultural Robotics ; Manufacturing; Agri-Tech',
                             'E-commerce; Grocery Delivery; FoodTech ; Retail & E-commerce',
                                       'Genetics; Preventive Health ; HealthTech; Genomics',
                                        'Digital Agriculture ; Digital Agriculture; AgTech',
                                       'EdTech; Coding; STEM Education ; Education; EdTech',
                      'Emergency Services; Digital Health ; HealthTech; Emergency Services',
                                              'Agriculture & AgTech ; Agriculture & AgTech',
                                                            'FinTech & DFS ; FinTech & DFS',
 ...
                              'Education Technology

In [ ]:
company_df["mapped_sector"] = (
    company_df["sector_mapping_text"]
    .apply(assign_sieve_sector)
)

print("Sector mapping completed successfully.")

Sector mapping completed successfully.


In [ ]:
sector_dictionary.keys()

dict_keys(['ATX', 'ETX', 'HTX', 'ERG', 'MFG', 'REC', 'FTX'])

In [ ]:
# Preview the first 30 mapped companies
company_df[
    ["sector_mapping_text", "mapped_sector"]
].head(30)

,sector_mapping_text,mapped_sector
0,Oil & Gas ; Oil & Gas; Fuel Retail,XSC
1,E-Pharmacy; Digital Pharmacy ; HealthTech; Digital Pharmacy,HTX
2,Agricultural Robotics ; Manufacturing; Agri-Tech,XSC
3,E-commerce; Grocery Delivery; FoodTech ; Retail & E-commerce,XSC
4,Genetics; Preventive Health ; HealthTech; Genomics,HTX
5,Digital Agriculture ; Digital Agriculture; AgTech,ATX
6,EdTech; Coding; STEM Education ; Education; EdTech,ETX
7,Emergency Services; Digital Health ; HealthTech; Emergency Services,HTX
8,Agriculture & AgTech ; Agriculture & AgTech,ATX
9,FinTech & DFS ; FinTech & DFS,FTX


In [ ]:
# Create Company Age

current_year = pd.Timestamp.today().year

company_df["company_age"] = current_year - company_df["year_founded"]

company_df.head()

,company_id,company_name,company_description,hq_country,hq_city,industry,employee_count,year_founded,products_and_services,operating_countries,sectors,sector_mapping_text,mapped_sector,company_age
0,COMP-000001,11 Plc,"Downstream petroleum marketing company engaged in fuel retail, lubricants, aviation fuel, and commercial fuel supply.",Nigeria,Lagos,Oil & Gas,"501–1,000",1979,Petrol stations; diesel; gasoline; aviation fuel; lubricants,Nigeria,Oil & Gas; Fuel Retail,Oil & Gas ; Oil & Gas; Fuel Retail,XSC,47
1,COMP-000002,3elagi,"Digital pharmacy platform providing online access to medicines, healthcare products, and beauty items with home delivery services.",Egypt,Cairo,E-Pharmacy; Digital Pharmacy,11–50,2017,Online pharmacy; medicine ordering; prescription fulfilment; healthcare products,Egypt,HealthTech; Digital Pharmacy,E-Pharmacy; Digital Pharmacy ; HealthTech; Digital Pharmacy,HTX,9
2,COMP-000003,3Farmate Robotics,Agricultural robotics company designing AI-powered autonomous farming equipment for precision agriculture and crop monitoring.,Ghana,Accra,Agricultural Robotics,2–10,2022,Autonomous farming robots; crop monitoring systems; AI agricultural platforms,Ghana,Manufacturing; Agri-Tech,Agricultural Robotics ; Manufacturing; Agri-Tech,XSC,4
3,COMP-000004,3lyna,On-demand grocery delivery platform connecting consumers with grocery retailers for home delivery.,Sudan,Khartoum,E-commerce; Grocery Delivery; FoodTech,2–10,2018,Grocery ordering; last-mile delivery; mobile commerce,Sudan,Retail & E-commerce,E-commerce; Grocery Delivery; FoodTech ; Retail & E-commerce,XSC,8
4,COMP-000005,4-DNA,"Health technology company using genetic intelligence to provide personalized health, fitness, nutrition, and risk insights.",Rwanda,Kigali,Genetics; Preventive Health,1–10,2020,DNA analysis; personalized health insights; genetic risk assessment tools.,Rwanda,HealthTech; Genomics,Genetics; Preventive Health ; HealthTech; Genomics,HTX,6


In [ ]:
company_df["employee_count"].unique()

<StringArray>
[         '501–1,000',              '11–50',               '2–10',
               '1–10',            'Unknown',             '51–200',
            '10,000+',       '5,001–10,000',          '1001–5000',
        '1,001–5,000',                '6–9',                '1–5',
            '101–250',             '1,200+',            '201–500',
             '1,000+',              '10–20',              '46297',
           '501–1000',            '15,000+',            '79,000+',
            '34,000+',            '90,000+',              '1000+',
            '10,001+',               '5–15',      '11–50 (Local)',
            '30,000+',              '11-50',            '251–500',
   '10,001+ (Global)',             '51-200',         '5001–10000',
              '10–50',      '10,000–25,000',           '270,000+',
           '300,000+',           '100,000+',            '40,000+',
 '4,000+ (Corporate)',      '10,000–15,000',               '500+',
        '2,001–5,000']
Length: 43, dtype: string

In [ ]:
# Standardize Employee Count
def standardize_employee_count(value):

    # Handle missing values
    if pd.isna(value):
        return "Unknown"

    value = str(value).strip()

    # Standardize formatting
    value = value.replace(",", "")
    value = value.replace("–", "-")
    value = re.sub(r"\s*\(.*?\)", "", value).strip()

    # Handle Unknown
    if value.lower() == "unknown":
        return "Unknown"

    # Handle values ending with +
    if value.endswith("+"):

        number = int(re.findall(r"\d+", value)[0])

        if number < 10:
            return "1–10"
        elif number < 50:
            return "11–50"
        elif number < 200:
            return "51–200"
        elif number < 500:
            return "201–500"
        elif number < 1000:
            return "501–1,000"
        elif number < 5000:
            return "1,001–5,000"
        elif number < 10000:
            return "5,001–10,000"
        else:
            return "10,000+"

    # Handle ranges
    elif "-" in value:

        lower, upper = map(int, value.split("-"))

        # Use upper bound
        number = upper

    # Handle exact numbers
    else:

        number = int(value)

    # Standard categories
    if number <= 10:
        return "1–10"
    elif number <= 50:
        return "11–50"
    elif number <= 200:
        return "51–200"
    elif number <= 500:
        return "201–500"
    elif number <= 1000:
        return "501–1,000"
    elif number <= 5000:
        return "1,001–5,000"
    elif number <= 10000:
        return "5,001–10,000"
    else:
        return "10,000+"

In [ ]:
# Create standardized employee count column
company_df["employee_count_standardized"] = (
    company_df["employee_count"]
    .apply(standardize_employee_count)
)

In [ ]:
company_df[
    ["employee_count", "employee_count_standardized"]
].drop_duplicates().sort_values("employee_count")

,employee_count,employee_count_standardized
92,"1,000+","1,001–5,000"
35,"1,001–5,000","1,001–5,000"
83,"1,200+","1,001–5,000"
18,"10,000+","10,000+"
1333,"10,000–15,000","10,000+"
905,"10,000–25,000","10,000+"
413,"10,001+","10,000+"
604,"10,001+ (Global)","10,000+"
1067,"100,000+","10,000+"
398,1000+,"1,001–5,000"


In [ ]:
company_df["employee_count_standardized"].unique()

array(['501–1,000', '11–50', '1–10', 'Unknown', '51–200', '10,000+',
       '5,001–10,000', '1,001–5,000', '201–500'], dtype=object)

In [ ]:
# Derive AIKYA Tier

def assign_aikya_tier(employee_count):

    # Handle unknown values
    if employee_count == "Unknown":
        return "Unknown"

    # Small Enterprise
    elif employee_count == "1–10":
        return "AIKYA HUSTLE"

    # Medium Enterprise
    elif employee_count == "11–50":
        return "AIKYA GROW"

    # Large Enterprise (51+ employees)
    else:
        return "AIKYA LEAD"


# Create AIKYA Tier column
company_df["aikya_tier"] = (
    company_df["employee_count_standardized"]
    .apply(assign_aikya_tier)
)

In [ ]:
company_df.head(5)

,company_id,company_name,company_description,hq_country,hq_city,industry,employee_count,year_founded,products_and_services,operating_countries,sectors,sector_mapping_text,mapped_sector,company_age,employee_count_standardized,aikya_tier
0,COMP-000001,11 Plc,"Downstream petroleum marketing company engaged in fuel retail, lubricants, aviation fuel, and commercial fuel supply.",Nigeria,Lagos,Oil & Gas,"501–1,000",1979,Petrol stations; diesel; gasoline; aviation fuel; lubricants,Nigeria,Oil & Gas; Fuel Retail,Oil & Gas ; Oil & Gas; Fuel Retail,XSC,47,"501–1,000",AIKYA LEAD
1,COMP-000002,3elagi,"Digital pharmacy platform providing online access to medicines, healthcare products, and beauty items with home delivery services.",Egypt,Cairo,E-Pharmacy; Digital Pharmacy,11–50,2017,Online pharmacy; medicine ordering; prescription fulfilment; healthcare products,Egypt,HealthTech; Digital Pharmacy,E-Pharmacy; Digital Pharmacy ; HealthTech; Digital Pharmacy,HTX,9,11–50,AIKYA GROW
2,COMP-000003,3Farmate Robotics,Agricultural robotics company designing AI-powered autonomous farming equipment for precision agriculture and crop monitoring.,Ghana,Accra,Agricultural Robotics,2–10,2022,Autonomous farming robots; crop monitoring systems; AI agricultural platforms,Ghana,Manufacturing; Agri-Tech,Agricultural Robotics ; Manufacturing; Agri-Tech,XSC,4,1–10,AIKYA HUSTLE
3,COMP-000004,3lyna,On-demand grocery delivery platform connecting consumers with grocery retailers for home delivery.,Sudan,Khartoum,E-commerce; Grocery Delivery; FoodTech,2–10,2018,Grocery ordering; last-mile delivery; mobile commerce,Sudan,Retail & E-commerce,E-commerce; Grocery Delivery; FoodTech ; Retail & E-commerce,XSC,8,1–10,AIKYA HUSTLE
4,COMP-000005,4-DNA,"Health technology company using genetic intelligence to provide personalized health, fitness, nutrition, and risk insights.",Rwanda,Kigali,Genetics; Preventive Health,1–10,2020,DNA analysis; personalized health insights; genetic risk assessment tools.,Rwanda,HealthTech; Genomics,Genetics; Preventive Health ; HealthTech; Genomics,HTX,6,1–10,AIKYA HUSTLE


In [ ]:
company_df.drop(columns=["sector_mapping_text"], inplace=True)

In [ ]:
# Reorder columns

company_df = company_df[
    [
        "company_id",
        "company_name",
        "company_description",
        "hq_country",
        "hq_city",
        "industry",
        "employee_count",
        "employee_count_standardized",
        "aikya_tier",
        "year_founded",
        "company_age",
        "products_and_services",
        "operating_countries",
        "sectors",
        "mapped_sector",
    ]
]

In [ ]:
company_df

,company_id,company_name,company_description,hq_country,hq_city,industry,employee_count,employee_count_standardized,aikya_tier,year_founded,company_age,products_and_services,operating_countries,sectors,mapped_sector
0,COMP-000001,11 Plc,"Downstream petroleum marketing company engaged in fuel retail, lubricants, aviation fuel, and commercial fuel supply.",Nigeria,Lagos,Oil & Gas,"501–1,000","501–1,000",AIKYA LEAD,1979,47,Petrol stations; diesel; gasoline; aviation fuel; lubricants,Nigeria,Oil & Gas; Fuel Retail,XSC
1,COMP-000002,3elagi,"Digital pharmacy platform providing online access to medicines, healthcare products, and beauty items with home delivery services.",Egypt,Cairo,E-Pharmacy; Digital Pharmacy,11–50,11–50,AIKYA GROW,2017,9,Online pharmacy; medicine ordering; prescription fulfilment; healthcare products,Egypt,HealthTech; Digital Pharmacy,HTX
2,COMP-000003,3Farmate Robotics,Agricultural robotics company designing AI-powered autonomous farming equipment for precision agriculture and crop monitoring.,Ghana,Accra,Agricultural Robotics,2–10,1–10,AIKYA HUSTLE,2022,4,Autonomous farming robots; crop monitoring systems; AI agricultural platforms,Ghana,Manufacturing; Agri-Tech,XSC
3,COMP-000004,3lyna,On-demand grocery delivery platform connecting consumers with grocery retailers for home delivery.,Sudan,Khartoum,E-commerce; Grocery Delivery; FoodTech,2–10,1–10,AIKYA HUSTLE,2018,8,Grocery ordering; last-mile delivery; mobile commerce,Sudan,Retail & E-commerce,XSC
4,COMP-000005,4-DNA,"Health technology company using genetic intelligence to provide personalized health, fitness, nutrition, and risk insights.",Rwanda,Kigali,Genetics; Preventive Health,1–10,1–10,AIKYA HUSTLE,2020,6,DNA analysis; personalized health insights; genetic risk assessment tools.,Rwanda,HealthTech; Genomics,HTX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1470,COMP-001471,Zulzi,On-demand grocery delivery marketplace providing consumers with same-day delivery from supermarkets and retailers.,South Africa,Johannesburg,E-commerce; Grocery Delivery; FoodTech,11–50,11–50,AIKYA GROW,2016,10,Grocery delivery; online supermarket; mobile shopping,South Africa,Retail & E-commerce; FoodTech,XSC
1471,COMP-001472,Zuri Health,"Digital health startup providing affordable healthcare services through mobile applications, websites, and SMS-based healthcare platforms.",Kenya,Nairobi,Telemedicine; Digital Health,11–50,11–50,AIKYA GROW,2021,5,Telemedicine; online consultations; SMS healthcare services; digital health access,Kenya; Ghana; Nigeria; Uganda; Zambia; Tanzania,HealthTech; Telemedicine,HTX
1472,COMP-001473,Zydii,Workforce development and digital transformation platform providing localized micro-learning content and professional training for individuals and organizations.,Kenya,Nairobi,JobTech; Corporate Learning; EdTech,11–50,11–50,AIKYA GROW,2017,9,WhatsApp-based micro-learning tracks; Zydii for Business platform; workplace skills training,Kenya; East Africa,Education; EdTech; JobTech,ETX
1473,COMP-001474,Zyptyk,"Digital health platform by Zomujo providing remote patient monitoring, chronic care management, and AI-powered mental health tools.",Ghana,Accra,Chronic Care; Mental Health; Analytics,11–50,11–50,AIKYA GROW,2020,6,Chronic disease tracking (diabetes/hypertension); peer support; ZypMind AI companion,Ghana,HealthTech; Digital Health,HTX


In [ ]:
# Create standardized company dataset

company_standardized = (
    company_df[
        [
            "company_id",
            "company_name",
            "company_description",
            "year_founded",
            "company_age",
            "hq_country",
            "hq_city",
            "operating_countries",
            "products_and_services",
            "employee_count_standardized",
            "aikya_tier",
            "mapped_sector",
        ]
    ]
    .rename(
        columns={
            "employee_count_standardized": "employee_count",
            "mapped_sector": "operational_sector",
        }
    )
)

# Display the standardized dataset
company_standardized.head()

,company_id,company_name,company_description,year_founded,company_age,hq_country,hq_city,operating_countries,products_and_services,employee_count,aikya_tier,operational_sector
0,COMP-000001,11 Plc,"Downstream petroleum marketing company engaged in fuel retail, lubricants, aviation fuel, and commercial fuel supply.",1979,47,Nigeria,Lagos,Nigeria,Petrol stations; diesel; gasoline; aviation fuel; lubricants,"501–1,000",AIKYA LEAD,XSC
1,COMP-000002,3elagi,"Digital pharmacy platform providing online access to medicines, healthcare products, and beauty items with home delivery services.",2017,9,Egypt,Cairo,Egypt,Online pharmacy; medicine ordering; prescription fulfilment; healthcare products,11–50,AIKYA GROW,HTX
2,COMP-000003,3Farmate Robotics,Agricultural robotics company designing AI-powered autonomous farming equipment for precision agriculture and crop monitoring.,2022,4,Ghana,Accra,Ghana,Autonomous farming robots; crop monitoring systems; AI agricultural platforms,1–10,AIKYA HUSTLE,XSC
3,COMP-000004,3lyna,On-demand grocery delivery platform connecting consumers with grocery retailers for home delivery.,2018,8,Sudan,Khartoum,Sudan,Grocery ordering; last-mile delivery; mobile commerce,1–10,AIKYA HUSTLE,XSC
4,COMP-000005,4-DNA,"Health technology company using genetic intelligence to provide personalized health, fitness, nutrition, and risk insights.",2020,6,Rwanda,Kigali,Rwanda,DNA analysis; personalized health insights; genetic risk assessment tools.,1–10,AIKYA HUSTLE,HTX


In [ ]:
# Export datasets to a single Excel workbook

output_file = "company_datasets.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    # Master dataset
    company_df.to_excel(
        writer,
        sheet_name="Company_Master",
        index=False
    )

    # Standardized dataset
    company_standardized.to_excel(
        writer,
        sheet_name="Company_Standardized",
        index=False
    )

print(f"Datasets successfully saved to '{output_file}'.")

Datasets successfully saved to 'company_datasets.xlsx'.


In [ ]:
from google.colab import files

files.download("company_datasets.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Baseline Model - Before Finetuning**

The modeling process begins with the company_standardized dataset, which contains the cleaned and standardized company records prepared for machine learning.

The dataset is first divided into two groups based on the operational sector label. Companies classified into a single operational sector (ATX, ETX, FTX, HTX, REC, ERG, and MFG) are separated from companies classified as Cross-Sector (XSC).

The single-sector companies form the modeling dataset and are split into training and testing subsets using train_test_split(). The training set (X_train, y_train) is used to train the baseline machine learning model, while the testing set (X_test, y_test) is used to evaluate its performance on unseen single-sector companies.

Companies initially labeled as XSC are excluded from the training process and retained in a separate validation dataset (validation_df). This dataset is not used to train the model. Instead, it serves as an independent validation set to assess how the trained classifier behaves when presented with companies operating across multiple sectors. This enables evaluation of the model's ability to generalize beyond the single-sector classes on which it was trained, without influencing the learning process itself.

Note: "If I train on pure sectors only, how does the model behave when it encounters mixed-sector companies?"

The machine learning model should answer "What is the dominant operational sector?" Then the business rules can answer "Is this actually cross-sector?"

In [ ]:
model_df = company_standardized[
    [
        "company_description",
        "products_and_services",
        "operational_sector"
    ]
].copy()
model_df.head()

,company_description,products_and_services,operational_sector
0,"Downstream petroleum marketing company engaged in fuel retail, lubricants, aviation fuel, and commercial fuel supply.",Petrol stations; diesel; gasoline; aviation fuel; lubricants,XSC
1,"Digital pharmacy platform providing online access to medicines, healthcare products, and beauty items with home delivery services.",Online pharmacy; medicine ordering; prescription fulfilment; healthcare products,HTX
2,Agricultural robotics company designing AI-powered autonomous farming equipment for precision agriculture and crop monitoring.,Autonomous farming robots; crop monitoring systems; AI agricultural platforms,XSC
3,On-demand grocery delivery platform connecting consumers with grocery retailers for home delivery.,Grocery ordering; last-mile delivery; mobile commerce,XSC
4,"Health technology company using genetic intelligence to provide personalized health, fitness, nutrition, and risk insights.",DNA analysis; personalized health insights; genetic risk assessment tools.,HTX


In [ ]:
# Exclude cross-sector companies from model training
single_sector_df = model_df[
    (~model_df["operational_sector"].isin(["XSC", "Unknown"]))
].copy()

validation_df=model_df[
    model_df["operational_sector"] == "XSC"
].copy()

# Define the input feature (combined company text)
X = single_sector_df["text"]

# Define the target variable (ground truth operational sector)
y = single_sector_df["operational_sector"]

# Split the dataset into training (80%) and testing (20%) subsets
# Stratification preserves the class distribution across both sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
# Initialize the TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 4),
    min_df=3,
    max_features=5000
)

In [ ]:
# Learn the vocabulary from the training data and convert it into TF-IDF features
X_train_tfidf = vectorizer.fit_transform(X_train)
# Convert the testing data using the same learned vocabulary
X_test_tfidf = vectorizer.transform(X_test)
# Convert the cross-sector validation data using the same learned vocabulary
X_validation_tfidf = vectorizer.transform(validation_df["text"])

print(f"Training matrix shape: {X_train_tfidf.shape}")
print(f"Testing matrix shape: {X_test_tfidf.shape}")
print(f"Validation matrix shape: {X_validation_tfidf.shape}")

Training matrix shape: (868, 2153)
Testing matrix shape: (218, 2153)
Validation matrix shape: (385, 2153)


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
y_pred = model.predict(X_test_tfidf)

In [ ]:
# Calculate the overall classification accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.2%}\n")

# Display precision, recall and F1-score for each sector
print(classification_report(y_test, y_pred))

Accuracy: 95.41%

              precision    recall  f1-score   support

         ATX       0.90      0.92      0.91        39
         ERG       1.00      0.90      0.95        20
         ETX       1.00      1.00      1.00        39
         FTX       0.93      1.00      0.96        38
         HTX       1.00      0.97      0.99        38
         MFG       1.00      0.69      0.82        13
         REC       0.91      1.00      0.95        31

    accuracy                           0.95       218
   macro avg       0.96      0.93      0.94       218
weighted avg       0.96      0.95      0.95       218



## Baseline Model Performance

### **Overall Accuracy: 95.41%**

This means:

- The model correctly classified **95.41%** of all companies in the testing dataset.
- Only **4.59%** of companies were assigned to an incorrect operational sector.
- This indicates that the baseline model has learned to distinguish between the seven operational sectors with a high degree of accuracy.



### **ATX (Agritech)**

- **Precision:** **0.90**
  - When the model predicts **ATX**, it is correct **90%** of the time.
  - Approximately **10%** of companies predicted as Agritech actually belong to another sector.

- **Recall:** **0.92**
  - The model successfully identifies **92%** of all Agritech companies.
  - Around **8%** of Agritech companies are classified into another sector.

- **F1-score:** **0.91**
  - This indicates a strong balance between correctly identifying Agritech companies and minimizing false Agritech predictions.


### **ERG (Energy)**

- **Precision:** **1.00**
  - Whenever the model predicts **ERG**, it is correct.
  - No non-Energy companies were incorrectly classified as Energy.

- **Recall:** **0.90**
  - The model correctly identifies **90%** of Energy companies.
  - Approximately **10%** of Energy companies are classified into another sector.

- **F1-score:** **0.95**
  - This indicates excellent classification performance with only a small number of missed Energy companies.


### **ETX (EdTech)**

- **Precision:** **1.00**
  - Every company predicted as **ETX** is actually an EdTech company.

- **Recall:** **1.00**
  - Every EdTech company in the testing dataset was successfully identified.

- **F1-score:** **1.00**
  - The model achieved perfect classification for the EdTech sector.

### **FTX (FinTech)**

- **Precision:** **0.93**
  - When the model predicts **FTX**, it is correct **93%** of the time.
  - Approximately **7%** of predicted FinTech companies belong to another sector.

- **Recall:** **1.00**
  - The model successfully identifies **every** FinTech company in the testing dataset.
  - No FinTech companies were missed.

- **F1-score:** **0.96**
  - This indicates excellent overall FinTech classification performance.


### **HTX (HealthTech)**

- **Precision:** **1.00**
  - Every company predicted as **HTX** is correctly classified.

- **Recall:** **0.97**
  - The model correctly identifies **97%** of HealthTech companies.
  - Only a very small number of HealthTech companies are classified elsewhere.

- **F1-score:** **0.99**
  - This indicates near-perfect HealthTech classification.


### **MFG (Manufacturing)**

- **Precision:** **1.00**
  - Whenever the model predicts **MFG**, it is correct.
  - No non-Manufacturing companies were incorrectly classified as Manufacturing.

- **Recall:** **0.69**
  - The model successfully identifies **69%** of Manufacturing companies.
  - Approximately **31%** of Manufacturing companies are classified into another sector.

  This suggests that many of the missed Manufacturing companies contain language that overlaps with other operational sectors, making them more difficult for the model to distinguish.

- **F1-score:** **0.82**
  - Manufacturing is currently the most challenging sector for the model and represents the greatest opportunity for future improvement.


### **REC (Retail & E-commerce)**

- **Precision:** **0.91**
  - When the model predicts **REC**, it is correct **91%** of the time.
  - Approximately **9%** of predicted Retail & E-commerce companies belong to another sector.

- **Recall:** **1.00**
  - Every Retail & E-commerce company in the testing dataset was successfully identified.
  - No Retail companies were missed.

- **F1-score:** **0.95**
  - This indicates excellent Retail & E-commerce classification performance.


## Macro Average

The **Macro Average** calculates the average performance across all sectors by giving **equal importance** to every operational sector, regardless of the number of companies in each class.

- **Precision:** **0.96**
- **Recall:** **0.93**
- **F1-score:** **0.94**

These results indicate that the model performs consistently across the different operational sectors, although Manufacturing slightly reduces the overall average because of its lower recall.


## Weighted Average

The **Weighted Average** calculates the average performance while accounting for the number of companies within each operational sector.

- **Precision:** **0.96**
- **Recall:** **0.95**
- **F1-score:** **0.95**

The weighted averages closely match the overall model accuracy, indicating that the classifier performs well across the dataset as a whole and that no large sector is performing poorly.


In [ ]:
import numpy as np
import pandas as pd

# Predict class probabilities for the XSC validation dataset
probabilities = model.predict_proba(X_validation_tfidf)

# Get the indices of the two highest probabilities
top2_idx = np.argsort(probabilities, axis=1)[:, -2:]

# Create a validation results dataframe
validation_results = validation_df.copy()

# Primary prediction
validation_results["primary_sector"] = [
    model.classes_[idx[1]] for idx in top2_idx
]

validation_results["primary_confidence"] = [
    probabilities[i, idx[1]]
    for i, idx in enumerate(top2_idx)
]

# Secondary prediction
validation_results["secondary_sector"] = [
    model.classes_[idx[0]] for idx in top2_idx
]

validation_results["secondary_confidence"] = [
    probabilities[i, idx[0]]
    for i, idx in enumerate(top2_idx)
]

# Display the results
validation_results[
    [
        "operational_sector",
        "primary_sector",
        "primary_confidence",
        "secondary_sector",
        "secondary_confidence"
    ]
].sort_values(
    by="primary_confidence",
    ascending=False
).head(20)

,operational_sector,primary_sector,primary_confidence,secondary_sector,secondary_confidence
329,XSC,HTX,0.929239,ETX,0.025760
164,XSC,ETX,0.902647,HTX,0.044687
363,XSC,HTX,0.901388,FTX,0.025604
1,XSC,ATX,0.892267,HTX,0.023399
321,XSC,ERG,0.877508,ATX,0.029999
170,XSC,ERG,0.875961,ATX,0.029371
182,XSC,ERG,0.866398,ATX,0.033130
82,XSC,ERG,0.865608,FTX,0.030859
255,XSC,HTX,0.862855,ATX,0.039499
111,XSC,ERG,0.860802,MFG,0.045565
